In [1]:
import numpy as np
import pandas as pd
 
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

def f1m(a, b):
    return f1_score(a, b, average="macro", labels=[0, 1], zero_division=0)
 
 
def dac_trung(s):
    t = s.split()
    d = [len(x) for x in t]
    return [len(t), len(s), float(np.mean(d)), max(d), min(d),
            sum(1 for x in t if "-" in x), len(set(t)), s.count("-")]

In [2]:
cot = ["nguon", "nhan", "nhan_goc", "cau"]
df = pd.read_csv("in_domain_train.tsv", sep="\t",
                  header=None, names=cot, quoting=3)

cau = df["cau"].astype(str).values
y = df["nhan"].values
X = np.array([dac_trung(s) for s in cau], dtype=float)

In [3]:
DO_SAU_LIST = [1, 2, 3, 5, 8, 12, 20, None]  
 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
 
ket_qua = []  # (do_sau, mean, std)
print(f"{'Do sau':<16}{'TB kiem dinh':>14}{'Do lech chuan':>16}{'5 fold':>28}")
for do_sau in DO_SAU_LIST:
    diem_5_fold = []
    for i_hoc, i_kiem in cv.split(X, y):
        m = DecisionTreeClassifier(max_depth=do_sau, random_state=0)
        m.fit(X[i_hoc], y[i_hoc])
        diem_5_fold.append(f1m(y[i_kiem], m.predict(X[i_kiem])))
    diem_5_fold = np.array(diem_5_fold)
    tb, std = diem_5_fold.mean(), diem_5_fold.std(ddof=1)
    ket_qua.append((do_sau, tb, std))
    nhan_do_sau = "khong gioi han" if do_sau is None else str(do_sau)
    chuoi_fold = " ".join(f"{x:.4f}" for x in diem_5_fold)
    print(f"{nhan_do_sau:<16}{tb:>14.4f}{std:>16.4f}   [{chuoi_fold}]")

Do sau            TB kiem dinh   Do lech chuan                      5 fold
1                       0.4133          0.0001   [0.4132 0.4134 0.4134 0.4132 0.4132]
2                       0.4133          0.0001   [0.4132 0.4134 0.4134 0.4132 0.4132]
3                       0.4139          0.0019   [0.4172 0.4134 0.4128 0.4128 0.4132]
5                       0.4199          0.0044   [0.4267 0.4186 0.4190 0.4147 0.4206]
8                       0.4329          0.0094   [0.4422 0.4355 0.4408 0.4219 0.4242]
12                      0.4629          0.0056   [0.4657 0.4680 0.4652 0.4537 0.4618]
20                      0.4916          0.0047   [0.4868 0.4875 0.4975 0.4910 0.4951]
khong gioi han          0.5017          0.0127   [0.5007 0.4926 0.5220 0.5039 0.4896]


In [4]:
def khoang(tb, std):
    return (tb - std, tb + std)

def chong_lan(a, b):
    return a[0] <= b[1] and b[0] <= a[1]

print("\nSo sanh doi mot: do sau nao phan biet duoc voi do sau nao")
so_cap_phan_biet_duoc = 0
so_cap_tong = 0
for i in range(len(ket_qua)):
    for j in range(i + 1, len(ket_qua)):
        ds_i, tb_i, std_i = ket_qua[i]
        ds_j, tb_j, std_j = ket_qua[j]
        kb_i, kb_j = khoang(tb_i, std_i), khoang(tb_j, std_j)
        phan_biet = not chong_lan(kb_i, kb_j)
        so_cap_tong += 1
        if phan_biet:
            so_cap_phan_biet_duoc += 1
        nhan_i = "khong gioi han" if ds_i is None else str(ds_i)
        nhan_j = "khong gioi han" if ds_j is None else str(ds_j)
        if phan_biet:
            print(f"  do sau {nhan_i:>15} vs {nhan_j:<15} -> PHAN BIET DUOC "
                  f"({tb_i:.4f} vs {tb_j:.4f})")
 
print(f"\nTong so cap phan biet duoc: {so_cap_phan_biet_duoc} / {so_cap_tong} cap co the")


So sanh doi mot: do sau nao phan biet duoc voi do sau nao
  do sau               1 vs 5               -> PHAN BIET DUOC (0.4133 vs 0.4199)
  do sau               1 vs 8               -> PHAN BIET DUOC (0.4133 vs 0.4329)
  do sau               1 vs 12              -> PHAN BIET DUOC (0.4133 vs 0.4629)
  do sau               1 vs 20              -> PHAN BIET DUOC (0.4133 vs 0.4916)
  do sau               1 vs khong gioi han  -> PHAN BIET DUOC (0.4133 vs 0.5017)
  do sau               2 vs 5               -> PHAN BIET DUOC (0.4133 vs 0.4199)
  do sau               2 vs 8               -> PHAN BIET DUOC (0.4133 vs 0.4329)
  do sau               2 vs 12              -> PHAN BIET DUOC (0.4133 vs 0.4629)
  do sau               2 vs 20              -> PHAN BIET DUOC (0.4133 vs 0.4916)
  do sau               2 vs khong gioi han  -> PHAN BIET DUOC (0.4133 vs 0.5017)
  do sau               3 vs 8               -> PHAN BIET DUOC (0.4139 vs 0.4329)
  do sau               3 vs 12              -> PHA

In [7]:
ket_qua_sap = sorted(ket_qua, key=lambda r: -r[1])
nhom = []
nhom_hien_tai = [ket_qua_sap[0]]
for r in ket_qua_sap[1:]:
    ds_truoc, tb_truoc, std_truoc = nhom_hien_tai[-1]
    ds_nay, tb_nay, std_nay = r
    if chong_lan(khoang(tb_truoc, std_truoc), khoang(tb_nay, std_nay)):
        nhom_hien_tai.append(r)
    else:
        nhom.append(nhom_hien_tai)
        nhom_hien_tai = [r]
nhom.append(nhom_hien_tai)
 
print(f"\nSo nhom thuc su phan biet duoc: {len(nhom)} (trong tong {len(ket_qua)} do sau)")
for idx, g in enumerate(nhom, 1):
    ten_ds = ["khong gioi han" if ds is None else str(ds) for ds, _, _ in g]
    print(f"  Nhom {idx}: do sau {ten_ds}")


So nhom thuc su phan biet duoc: 3 (trong tong 8 do sau)
  Nhom 1: do sau ['khong gioi han', '20']
  Nhom 2: do sau ['12']
  Nhom 3: do sau ['8', '5', '3', '1', '2']


In [6]:
d2 = next(r for r in ket_qua if r[0] == 2)
d8 = next(r for r in ket_qua if r[0] == 8)
print(f"\n=== Do sau 2 vs do sau 8 ===")
print(f"  do sau 2: TB={d2[1]:.4f}  std={d2[2]:.4f}  khoang=[{khoang(d2[1], d2[2])[0]:.4f}, {khoang(d2[1], d2[2])[1]:.4f}]")
print(f"  do sau 8: TB={d8[1]:.4f}  std={d8[2]:.4f}  khoang=[{khoang(d8[1], d8[2])[0]:.4f}, {khoang(d8[1], d8[2])[1]:.4f}]")
if chong_lan(khoang(d2[1], d2[2]), khoang(d8[1], d8[2])):
    print("  -> Khoang dao dong CHONG LAN: khong the noi do sau nao thang, "
          "chenh lech nam trong nhieu.")
else:
    thang = "do sau 2" if d2[1] > d8[1] else "do sau 8"
    print(f"  -> Khoang dao dong KHONG chong lan: {thang} thuc su thang.")


=== Do sau 2 vs do sau 8 ===
  do sau 2: TB=0.4133  std=0.0001  khoang=[0.4132, 0.4134]
  do sau 8: TB=0.4329  std=0.0094  khoang=[0.4235, 0.4423]
  -> Khoang dao dong KHONG chong lan: do sau 8 thuc su thang.
